In [1]:
import sys; sys.path.insert(0, "../src")
import water_utils as wu

con = wu.connect()          # opens data/processed/water.db
Q = wu.load_queries()       # reads every query from sql/queries.sql

for name, sql in Q.items():
    print("=" * 90)
    print("QUERY:", name)
    print(sql)                              # shows the SQL code
    result = wu.run(con, name, Q)           # runs it
    if result is None:
        print("-> views created (no table to show)")
    else:
        display(result.head(20))            # shows the result table

QUERY: q01_row_counts
-- Q1. How many rows are in each table? (first row-count check)
SELECT 'bills_raw'      AS table_name, COUNT(*) AS n_rows FROM bills_raw
UNION ALL SELECT 'developments',   COUNT(*) FROM developments
UNION ALL SELECT 'complaints_raw', COUNT(*) FROM complaints_raw
UNION ALL SELECT 'weather_daily',  COUNT(*) FROM weather_daily;


,table_name,n_rows
0,bills_raw,66432
1,developments,341
2,complaints_raw,11252
3,weather_daily,4534


QUERY: q02_bills_by_borough
-- Q2. GROUP BY: size of the meter data per borough, and how often meters were NOT read.
SELECT  UPPER(TRIM(borough))                                   AS borough,
        COUNT(*)                                               AS bills,
        COUNT(DISTINCT tds)                                    AS developments,
        COUNT(DISTINCT meter_number)                           AS meters,
        ROUND(100.0 * SUM(CASE WHEN UPPER(estimated) = 'Y' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                               AS pct_estimated,
        ROUND(SUM(CAST(consumption_hcf AS REAL)), 0)           AS total_hcf
FROM    bills_raw
GROUP BY UPPER(TRIM(borough))
ORDER BY bills DESC;


,borough,bills,developments,meters,pct_estimated,total_hcf
0,FHA,28375,11,404,17.6,336264.0
1,BROOKLYN,13100,91,270,16.4,172929305.0
2,MANHATTAN,11494,99,229,11.5,167412910.0
3,BRONX,6943,80,146,17.8,135764174.0
4,QUEENS,6156,21,124,21.9,49321492.0
5,STATEN ISLAND,264,9,3,0.8,14807432.0
6,NON DEVELOPMENT FACILITY,100,1,2,2.0,1013.0


QUERY: q03_filter_largest_actual_2024
-- Q3. WHERE filter + ORDER BY: the 20 largest ACTUAL (not estimated) meter bills in 2024.
SELECT  development_name, borough, meter_number, revenue_month,
        CAST(days AS INTEGER)            AS days,
        CAST(consumption_hcf AS REAL)    AS consumption_hcf,
        ROUND(CAST(consumption_hcf AS REAL) / CAST(days AS REAL), 1) AS hcf_per_day
FROM    bills_raw
WHERE   UPPER(estimated) = 'N'
  AND   revenue_month BETWEEN '2024-01' AND '2024-12'
  AND   CAST(days AS REAL) > 0
ORDER BY CAST(consumption_hcf AS REAL) DESC
LIMIT 20;


,development_name,borough,meter_number,revenue_month,days,consumption_hcf,hcf_per_day
0,SEWARD PARK EXTENSION,MANHATTAN,E14081316,2024-03,102,6448.0,63.2
1,MELTZER TOWER,MANHATTAN,9020850,2024-06,92,6310.0,68.6
2,MELTZER TOWER,MANHATTAN,9020850,2024-09,91,6125.0,67.3
3,SEWARD PARK EXTENSION,MANHATTAN,E14081316,2024-06,82,5662.0,69.0
4,MELTZER TOWER,MANHATTAN,9020850,2024-12,91,5591.0,61.4
5,SEWARD PARK EXTENSION,MANHATTAN,E14122365,2024-12,91,5525.0,60.7
6,MELTZER TOWER,MANHATTAN,9020850,2024-03,50,3847.0,76.9
7,SEWARD PARK EXTENSION,MANHATTAN,K12447340,2024-01,34,3430.0,100.9
8,MELTZER TOWER,MANHATTAN,9020850,2024-01,42,3311.0,78.8
9,SEWARD PARK EXTENSION,MANHATTAN,K12447340,2024-10,32,3280.0,102.5


... (16 outputs left out)

QUERY: v01_monthly_views
-- V1. Views that turn daily weather and raw 311 counts into monthly tables.
DROP VIEW IF EXISTS weather_monthly;
CREATE VIEW weather_monthly AS
SELECT  SUBSTR(date, 1, 7)                                          AS month,
        ROUND(AVG(temperature_2m_mean), 2)                          AS temp_mean,
        MAX(temperature_2m_max)                                     AS temp_max,
        MIN(temperature_2m_min)                                     AS temp_min,
        ROUND(SUM(precipitation_sum), 1)                            AS precip_mm,
        ROUND(SUM(snowfall_sum), 1)                                 AS snow_cm,
        SUM(CASE WHEN temperature_2m_min < -5 THEN 1 ELSE 0 END)    AS freeze_days,
        SUM(CASE WHEN temperature_2m_max > 30 THEN 1 ELSE 0 END)    AS hot_days
FROM    weather_daily
GROUP BY SUBSTR(date, 1, 7);

DROP VIEW IF EXISTS complaints_monthly;
-- 311 complaint groups (descriptor list checked with q11):
--   supply disruption = no wa

,tds,revenue_month,borough,n_meters,n_est_meters,n_zero_meters,hcf,hcf_per_day,charges,development,...,snow_cm,freeze_days,hot_days,complaints_total,c_supply,c_no_water,c_low_pressure,c_main_break,c_leak,c_hydrant
0,1,2021-01,MANHATTAN,1,0,0,14.7,0.49,38.07,FIRST HOUSES,...,6.1,5,0,465,52,24,3,25,97,229
1,1,2021-02,MANHATTAN,1,0,1,0.0,0.00,40.61,FIRST HOUSES,...,57.8,12,0,293,36,13,4,19,64,96
2,1,2021-03,MANHATTAN,1,0,1,0.0,0.00,38.07,FIRST HOUSES,...,0.5,5,0,320,26,7,8,11,82,139
3,1,2021-04,MANHATTAN,1,0,1,0.0,0.00,36.80,FIRST HOUSES,...,0.0,0,0,351,24,18,0,6,65,197
4,1,2021-05,MANHATTAN,1,0,1,0.0,0.00,40.61,FIRST HOUSES,...,0.0,0,2,400,31,12,3,16,47,270
5,1,2021-06,MANHATTAN,1,0,1,0.0,0.00,38.07,FIRST HOUSES,...,0.0,0,9,1466,23,9,5,9,75,1292
6,1,2021-07,MANHATTAN,1,0,1,0.0,0.00,36.80,FIRST HOUSES,...,0.0,0,8,1601,39,22,5,12,95,1395
7,1,2021-08,MANHATTAN,1,0,1,0.0,0.00,41.88,FIRST HOUSES,...,0.0,0,9,1350,29,9,3,17,81,1100
8,1,2021-09,MANHATTAN,1,0,1,0.0,0.00,33.00,FIRST HOUSES,...,0.0,0,0,545,43,23,2,18,70,358
9,1,2021-10,MANHATTAN,1,0,1,0.0,0.00,38.07,FIRST HOUSES,...,0.0,0,0,451,26,8,8,10,54,321


QUERY: q13_panel_join_check
-- Q13. Row-count check for the panel: the joins must not add or lose rows.
WITH dev_month AS (
    SELECT tds, revenue_month, MIN(borough) AS borough FROM bills_clean GROUP BY tds, revenue_month
)
SELECT  (SELECT COUNT(*) FROM dev_month)                                   AS dev_months,
        COUNT(*)                                                           AS after_joins,
        SUM(CASE WHEN w.month   IS NULL THEN 1 ELSE 0 END)                 AS no_weather_match,
        SUM(CASE WHEN c.borough IS NULL THEN 1 ELSE 0 END)                 AS no_311_match,
        SUM(CASE WHEN d.tds     IS NULL THEN 1 ELSE 0 END)                 AS no_devbook_match
FROM    dev_month AS m
LEFT JOIN developments       AS d ON d.tds = m.tds
LEFT JOIN weather_monthly    AS w ON w.month = m.revenue_month
LEFT JOIN complaints_monthly AS c ON c.borough = m.borough AND c.month = m.revenue_month;


,dev_months,after_joins,no_weather_match,no_311_match,no_devbook_match
0,8752,8752,0,0,23
